# April run — smoke test

Exercises **every stage of the April pipeline end-to-end at tiny scope**
(1 day, both symbols, full predictor/class spec, 5 training epochs) so a
broken step fails here in minutes instead of hours into the real run.
Checks per symbol:

1. data discovery + per-symbol config construction (filter ON)
2. distributions stage → **10 `SEQ_DISTR_*` + 50 `CLS_DISTR_*_{cls}`** files
3. ensemble stage → **25 `ENS_TD_*`** files
4. training harness → 2 result files + 4 charts for one model

Everything runs in `outputs/april_smoke/` — the real `outputs/april/` and
`configs/april_*.yaml` are untouched. Expected runtime: ~10–20 min on the
compute box (dominated by featurizing 1 day per symbol).

This validates **plumbing, not numbers** — numerical correctness is covered
by the byte-equivalence suites you already ran (`verify_cls_v2`,
`verify_ensemble_reference`, `verify_against_baseline`).

In [ ]:
import shutil, sys, time, pathlib
ROOT = pathlib.Path.cwd()
for p in (ROOT, ROOT / "scripts", ROOT / "tests", ROOT / "TrainingDistributions"):
    sys.path.insert(0, str(p))

from run_april import (find_data_dir, april_dates, detect_pattern,
                       DIST_PREDICTORS, CLS_NAMES, PREDICTORS)
from pipeline.config import RunConfig

SYMBOLS = ["NVDA", "INTC"]
SMOKE = ROOT / "outputs" / "april_smoke"
shutil.rmtree(SMOKE, ignore_errors=True)

data_dir = find_data_dir()
pattern = detect_pattern(data_dir)
dates = april_dates(data_dir, pattern)[:1]          # 1 day = tiny scope
print(f"data: {data_dir}  (pattern: {pattern})")
print(f"smoke day: {dates}")

def smoke_config(symbol):
    """Same shape as run_april.make_config, redirected to the smoke dir."""
    cfg = RunConfig()
    cfg.data.symbol = symbol
    cfg.data.data_path = str(data_dir)
    cfg.data.dates = list(dates)
    cfg.data.file_pattern = pattern
    cfg.data.instrument_filter = True
    cfg.distributions.predictors = list(DIST_PREDICTORS)
    cfg.distributions.class_names = list(CLS_NAMES)
    cfg.distributions.class_values = [-1, 0, 1]
    cfg.distributions.output_dir = str((SMOKE / symbol).relative_to(ROOT))
    cfg.featurize.cache_dir = str((SMOKE / symbol / "feature_cache").relative_to(ROOT))
    cfg.ensemble.output_dir = str((SMOKE / symbol / "ensemble").relative_to(ROOT))
    cfg.training.model_dir = str((SMOKE / symbol / "models").relative_to(ROOT))
    (SMOKE / symbol).mkdir(parents=True, exist_ok=True)
    cfg.save(SMOKE / f"{symbol.lower()}.yaml")
    return cfg

configs = {s: smoke_config(s) for s in SYMBOLS}
print("PASS  configs built (filter ON, 10 predictors, 5 classes)")

In [ ]:
from pipeline.runner import run

for symbol, cfg in configs.items():
    t0 = time.time()
    run(cfg, run_id=f"smoke-dist-{symbol}")
    out = SMOKE / symbol
    n_seq = len(list(out.glob("SEQ_DISTR_*")))
    n_cls = len(list(out.glob("CLS_DISTR_*")))
    v2_named = len(list(out.glob(f"CLS_DISTR_{symbol}__log_mid-*_202504_*")))
    assert n_seq == 10, f"{symbol}: expected 10 SEQ files, got {n_seq}"
    assert n_cls == 50, f"{symbol}: expected 50 CLS files, got {n_cls}"
    assert v2_named == 50, f"{symbol}: CLS files not in v2 naming"
    print(f"PASS  {symbol} distributions: 10 SEQ + 50 CLS (v2 names) "
          f"in {time.time()-t0:.0f}s")

In [ ]:
from pipeline.ensemble import run_ensemble

for symbol, cfg in configs.items():
    t0 = time.time()
    outputs = run_ensemble(cfg, run_id=f"smoke-ens-{symbol}")
    n_ens = len(list((SMOKE / symbol / "ensemble").glob("ENS_TD_*")))
    assert n_ens == 25, f"{symbol}: expected 25 ENS_TD files, got {n_ens}"
    print(f"PASS  {symbol} ensemble: 25 ENS_TD files in {time.time()-t0:.0f}s")

In [ ]:
from train_kraus_baseline import run_one

for symbol in SYMBOLS:
    distr_dir = SMOKE / symbol
    out_dir = distr_dir / "models"
    out_dir.mkdir(parents=True, exist_ok=True)
    r = run_one(PREDICTORS[0], distr_dir, out_dir, epochs=5, seed=0,
                symbol=symbol, n_qubits=3)
    assert pathlib.Path(r["model_pickle"]).exists()
    assert pathlib.Path(r["weights"]).exists()
    assert len(r["plots"]) == 4, f"expected 4 charts, got {len(r['plots'])}"
    print(f"PASS  {symbol} training: 2 result files + 4 charts "
          f"({r['examples']} examples, {r['train_seconds']:.0f}s)")

print("\nALL SMOKE CHECKS PASSED — every April-run stage works end to end.")
print("Safe to run april_run.ipynb at full scope.")